# Notebook 03 (Participant): Add a New Problem Scaffold

You will implement a battery cold-plate problem contract and evaluate whether it is benchmark-ready.


**Edit-safe start:** this notebook opens from GitHub in read-only source mode. Use **File -> Save a copy in Drive** before running edits so your changes stay in your own workspace.


## Notebook map

This notebook is written as a standalone lab chapter:
- context first,
- implementation second,
- interpretation third.

If you are following asynchronously, run cells in order and use the success checks to validate each stage before moving on.


## Standalone guide

This chapter is about benchmark design quality, not model training speed.


## What makes a new problem benchmark-ready

A publishable benchmark needs explicit representation, constraints, objectives, simulator semantics, and reproducibility metadata.


In [ ]:
# Colab/local dependency bootstrap
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
FORCE_INSTALL = False  # Set True to force reinstall outside Colab
PACKAGES = ['engibench[beams2d]', 'matplotlib', 'gymnasium']

if IN_COLAB or FORCE_INSTALL:
    print('Installing dependencies...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *PACKAGES])
    print('Dependency install complete.')
else:
    print('Skipping install (using current environment).')


### Step 1 - Import scaffold dependencies

Ensure all required interfaces are visible before class implementation.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Annotated

import numpy as np
from gymnasium import spaces

from engibench.constraint import bounded
from engibench.constraint import constraint
from engibench.core import ObjectiveDirection
from engibench.core import OptiStep
from engibench.core import Problem


### Step 2 - Implement battery cold-plate problem contract (TODO)

Complete each required method with deterministic behavior and clear failure messages.


In [ ]:
class BatteryColdPlate2DProblem(Problem[np.ndarray]):
    """Scaffold for a battery cold-plate topology problem (not currently in EngiBench)."""

    version = 0
    objectives = (
        ("max_temperature_c", ObjectiveDirection.MINIMIZE),
        ("flow_penalty", ObjectiveDirection.MINIMIZE),
    )

    @dataclass
    class Conditions:
        heat_load_left: Annotated[float, bounded(lower=0.1, upper=2.0)] = 1.0
        heat_load_right: Annotated[float, bounded(lower=0.1, upper=2.0)] = 1.0
        inlet_temp_c: Annotated[float, bounded(lower=10.0, upper=40.0)] = 25.0
        flow_budget: Annotated[float, bounded(lower=0.15, upper=0.65)] = 0.35

    @dataclass
    class Config(Conditions):
        resolution: Annotated[int, bounded(lower=16, upper=96)] = 32
        max_iter: Annotated[int, bounded(lower=1, upper=200)] = 30
        solver_iters: Annotated[int, bounded(lower=20, upper=500)] = 120
        min_channel_fraction: Annotated[float, bounded(lower=0.05, upper=0.60)] = 0.18
        max_channel_fraction: Annotated[float, bounded(lower=0.10, upper=0.85)] = 0.55
        max_edge_density: Annotated[float, bounded(lower=0.01, upper=1.00)] = 0.28

    dataset_id = "IDEALLab/battery_cold_plate_2d_v0"  # placeholder for future dataset integration
    container_id = None

    def __init__(self, seed: int = 0, **kwargs):
        super().__init__(seed=seed)
        self.config = self.Config(**kwargs)
        self.conditions = self.Conditions(
            heat_load_left=self.config.heat_load_left,
            heat_load_right=self.config.heat_load_right,
            inlet_temp_c=self.config.inlet_temp_c,
            flow_budget=self.config.flow_budget,
        )
        self.design_space = spaces.Box(
            low=0.0,
            high=1.0,
            shape=(self.config.resolution, self.config.resolution),
            dtype=np.float32,
        )

        # TODO 1: add at least two design constraints using @constraint
        # Suggested:
        # - channel fraction between min_channel_fraction and max_channel_fraction
        # - edge-density/manufacturability bound using max_edge_density
        raise NotImplementedError('Implement design constraints and assign self.design_constraints')

    def _heat_map(self, cfg: dict) -> np.ndarray:
        # TODO 2: implement two gaussian heat sources (left/right battery modules)
        raise NotImplementedError('Implement _heat_map')

    def _solve_temperature(self, conductivity: np.ndarray, heat: np.ndarray, inlet_temp: float, n_iter: int) -> np.ndarray:
        # TODO 3: implement iterative finite-difference temperature solver
        # Boundary suggestions:
        # - left boundary fixed at inlet_temp
        # - right boundary convective mix to ambient
        # - top/bottom insulated copy
        raise NotImplementedError('Implement _solve_temperature')

    def simulate(self, design: np.ndarray, config: dict | None = None) -> np.ndarray:
        # TODO 4: implement two-objective evaluation
        # Objective 1: max temperature [C]
        # Objective 2: flow_penalty (channel-fraction mismatch + roughness + mild temperature uniformity term)
        raise NotImplementedError('Implement simulate')

    def optimize(self, starting_point: np.ndarray, config: dict | None = None):
        # TODO 5: implement a simple iterative optimizer and return (design, list[OptiStep])
        # Keep it deterministic and lightweight for workshop runtime.
        raise NotImplementedError('Implement optimize')

    def render(self, design: np.ndarray, *, open_window: bool = False):
        import matplotlib.pyplot as plt

        # Visual semantics:
        # - solid/channel map: 1.0 means conductive solid, 0.0 means coolant channel
        # - conductivity map: effective thermal conductivity used by solver
        # - heat map: imposed module heat loads (problem conditions)
        # - temperature map: solved steady-state thermal field
        x = np.clip(design.astype(np.float32), 0.0, 1.0)
        channel = 1.0 - x

        cfg = self.config.__dict__
        k_channel = 0.25
        k_solid = 4.5
        conductivity = k_channel + x * (k_solid - k_channel)
        heat = self._heat_map(cfg)
        temperature = self._solve_temperature(conductivity, heat, cfg["inlet_temp_c"], cfg["solver_iters"])

        fig, axes = plt.subplots(1, 4, figsize=(16, 4))

        im0 = axes[0].imshow(x, cmap="gray", vmin=0, vmax=1)
        axes[0].set_title("Design map\n(1=solid, 0=channel)")
        axes[0].axis("off")
        fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

        im1 = axes[1].imshow(conductivity, cmap="cividis")
        axes[1].set_title("Conductivity field")
        axes[1].axis("off")
        fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

        im2 = axes[2].imshow(heat, cmap="magma")
        axes[2].set_title("Heat-load map")
        axes[2].axis("off")
        fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

        im3 = axes[3].imshow(temperature, cmap="inferno")
        axes[3].set_title("Temperature field [C]")
        axes[3].axis("off")
        fig.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)

        channel_fraction = float(np.mean(channel))
        edge_density = float(np.mean(np.abs(np.diff(channel, axis=0))) + np.mean(np.abs(np.diff(channel, axis=1))))
        fig.suptitle(
            f"channel_fraction={channel_fraction:.3f}, edge_density={edge_density:.3f}, "
            f"max_temp={float(np.max(temperature)):.2f}C",
            y=1.03,
        )

        fig.tight_layout()
        if open_window:
            plt.show()
        return fig, axes

    def random_design(self):
        # TODO 6: implement smooth random design initialization
        raise NotImplementedError('Implement random_design')


### Step 3 - Smoke-test your scaffold

Run minimal checks to verify interface consistency and simulator behavior.


Use the multi-panel render to read **where heat enters**, **how material is distributed**, and **where thermal bottlenecks remain**.


In [ ]:
# Run this cell after finishing TODOs in BatteryColdPlate2DProblem
problem = BatteryColdPlate2DProblem(
    seed=42,
    resolution=32,
    max_iter=20,
    heat_load_left=1.4,
    heat_load_right=1.1,
    inlet_temp_c=24.0,
    flow_budget=0.33,
)
start, _ = problem.random_design()

cfg = {
    'heat_load_left': 1.4,
    'heat_load_right': 1.1,
    'inlet_temp_c': 24.0,
    'flow_budget': 0.33,
    'resolution': 32,
    'max_iter': 20,
    'solver_iters': 100,
    'min_channel_fraction': 0.18,
    'max_channel_fraction': 0.55,
    'max_edge_density': 0.35,
}

print('design space:', problem.design_space)
print('objectives:', problem.objectives)
print('conditions:', problem.conditions)

viol = problem.check_constraints(start, config=cfg)
print('constraint violations:', len(viol))

obj0 = problem.simulate(start, config=cfg)
opt_design, history = problem.optimize(start, config=cfg)
objf = problem.simulate(opt_design, config=cfg)

print('initial objectives [max_temp_c, flow_penalty]:', obj0.tolist())
print('final objectives   [max_temp_c, flow_penalty]:', objf.tolist())
print('optimization steps:', len(history))

problem.render(opt_design)


print('How to read plots: design(1=solid,0=channel) | conductivity | heat-load | temperature')


## Mapping to real EngiBench contributions

Translate this battery cold-plate scaffold into domain-specific simulators and datasets with documented assumptions.


## Contribution checklist

Before proposing a new problem, verify data provenance, split policy, evaluation protocol, and reporting templates.


## Troubleshooting

If a section fails, do not continue downstream. Fix locally first, then rerun the section and its immediate checks.
This notebook is intentionally staged so failures are localized.


## Takeaways

Before closing, record three points:
1. What conclusion is directly supported by your metrics?
2. What remains uncertain (and why)?
3. What extra experiment would you run next to reduce that uncertainty?
